# TP 4 (Transformer) - Assembler et entrainer le modele complet

**Cours NLP et Transformers - CY Tech**

On reunit toutes les briques des TP precedents : embeddings de token et de
position, attention multi-tetes, blocs residuels avec LayerNorm et MLP, puis
unembedding et Softmax.

Resultat attendu : une perte d'entrainement et de validation qui diminue, puis
environ 2000 caracteres generes dans un style progressivement reconnaissable,
sans exiger un anglais correct. Un GPU accelere beaucoup l'entrainement.

## Etape 0 - Preparer le notebook

Hyperparametres et constantes. Sur CPU, on peut reduire `max_iters` pour un
premier test.

In [1]:
import urllib.request

import torch
import torch.nn as nn
from torch.nn import functional as F

batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

assert n_embd % n_head == 0
torch.manual_seed(1337)
print("device :", device)

device : cpu


## Donnees

On reutilise le telechargement, l'encodage et le decoupage train / validation du
TP 1.

In [2]:
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "input.txt",
)
with open("input.txt", "r", encoding="utf-8") as fichier:
    texte = fichier.read()

caracteres = sorted(list(set(texte)))
taille_vocabulaire = len(caracteres)
stoi = {c: i for i, c in enumerate(caracteres)}
itos = {i: c for i, c in enumerate(caracteres)}
encoder = lambda chaine: [stoi[c] for c in chaine]
decoder = lambda identifiants: "".join([itos[i] for i in identifiants])

donnees = torch.tensor(encoder(texte), dtype=torch.long)
n = int(0.9 * len(donnees))
donnees_train = donnees[:n]
donnees_val = donnees[n:]


def get_batch(partie):
    source = donnees_train if partie == "train" else donnees_val
    indices = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in indices])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in indices])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimer_perte():
    resultat = {}
    modele.eval()
    for partie in ["train", "val"]:
        pertes = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(partie)
            _, perte = modele(x, y)
            pertes[k] = perte.item()
        resultat[partie] = pertes.mean().item()
    modele.train()
    return resultat

## Etape 1 - Une tete d'attention

Identique au TP 2, avec en plus un masque causal enregistre comme buffer et un
dropout sur les poids.

In [3]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        poids = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        poids = poids.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        poids = F.softmax(poids, dim=-1)
        poids = self.dropout(poids)
        v = self.value(x)
        return poids @ v

## Etape 2 - Combiner plusieurs tetes

Chaque tete produit `head_size` composantes. La concatenation reconstitue la
dimension d'embedding, puis une projection melange les informations des tetes.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, nombre_tetes, head_size):
        super().__init__()
        self.tetes = nn.ModuleList([Head(head_size) for _ in range(nombre_tetes)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        sortie = torch.cat([tete(x) for tete in self.tetes], dim=-1)
        return self.dropout(self.projection(sortie))

## Etape 3 - Assembler un bloc

Le bloc alterne communication (attention) et calcul individuel (feed-forward).
Chaque sous-couche est precedee d'une LayerNorm et suivie d'une addition
residuelle.

In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.reseau = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.reseau(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.attention = MultiHeadAttention(n_head, head_size)
        self.feed_forward = FeedForward(n_embd)
        self.norme1 = nn.LayerNorm(n_embd)
        self.norme2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.attention(self.norme1(x))
        x = x + self.feed_forward(self.norme2(x))
        return x

## Etape 4 - Definir les couches du modele

Le modele separe l'embedding du token, l'embedding de position et l'unembedding
final vers les logits du vocabulaire. `lm_head` n'est pas l'embedding : c'est la
projection inverse, de l'espace contextualise vers les tokens.

In [6]:
class ModeleLangage(nn.Module):
    def __init__(self):
        super().__init__()
        self.table_embedding_token = nn.Embedding(taille_vocabulaire, n_embd)
        self.table_embedding_position = nn.Embedding(block_size, n_embd)
        self.blocs = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.norme_finale = nn.LayerNorm(n_embd)
        self.tete_sortie = nn.Linear(n_embd, taille_vocabulaire)

    def forward(self, indices, cibles=None):
        B, T = indices.shape
        emb_token = self.table_embedding_token(indices)
        emb_position = self.table_embedding_position(torch.arange(T, device=device))
        x = emb_token + emb_position
        x = self.blocs(x)
        x = self.norme_finale(x)
        logits = self.tete_sortie(x)

        if cibles is None:
            perte = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            cibles = cibles.view(B * T)
            perte = F.cross_entropy(logits, cibles)
        return logits, perte

    def generer(self, indices, nombre_tokens):
        for _ in range(nombre_tokens):
            indices_tronques = indices[:, -block_size:]
            logits, _ = self(indices_tronques)
            logits = logits[:, -1, :]
            probabilites = F.softmax(logits, dim=-1)
            indice_suivant = torch.multinomial(probabilites, num_samples=1)
            indices = torch.cat((indices, indice_suivant), dim=1)
        return indices

## Etapes 5 a 8 - Instancier, evaluer et entrainer

L'addition des deux embeddings fonctionne par broadcasting : l'embedding de
position (T, n_embd) s'ajoute a chaque sequence du batch. On tronque le contexte
a `block_size` pendant la generation car le modele ne sait traiter que cette
longueur.

In [7]:
modele = ModeleLangage().to(device)
print(round(sum(p.numel() for p in modele.parameters()) / 1e6, 3), "M parametres")

optimiseur = torch.optim.AdamW(modele.parameters(), lr=learning_rate)

for iteration in range(max_iters):
    if iteration % eval_interval == 0 or iteration == max_iters - 1:
        pertes = estimer_perte()
        print("pas", iteration,
              "| perte train", round(pertes["train"], 4),
              "| perte val", round(pertes["val"], 4))
    x, y = get_batch("train")
    logits, perte = modele(x, y)
    optimiseur.zero_grad(set_to_none=True)
    perte.backward()
    optimiseur.step()

0.21 M parametres


pas 0 | perte train 4.4109 | perte val 4.4016


pas 100 | perte train 2.6511 | perte val 2.6604


pas 200 | perte train 2.4941 | perte val 2.4902


pas 300 | perte train 2.3902 | perte val 2.4019


...

pas 4600 | perte train 1.6822 | perte val 1.8173


pas 4700 | perte train 1.6759 | perte val 1.8248


pas 4800 | perte train 1.6628 | perte val 1.832


pas 4900 | perte train 1.6669 | perte val 1.8296


pas 4999 | perte train 1.6624 | perte val 1.8165


## Etape 9 - Generer, puis experimenter

On part d'un contexte vide et on genere environ 2000 caracteres. Une generation
imparfaite est normale : l'objectif est de relier le resultat aux briques.

Pistes a tester :
- comparer le texte avant et apres entrainement ;
- tracer ou recopier les pertes train / validation ;
- passer `block_size` de 32 a 64 ;
- passer `n_head` de 1 a 4 a `n_embd` constant ;
- augmenter `dropout` si un sur-apprentissage apparait.

In [8]:
contexte = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decoder(modele.generer(contexte, 2000)[0].tolist()))


FlY BARD ELLO:
Let the destiviniter against there. I wome, gatart I am, it
forturatall me fore to galls; arth of those curth greation.

ARCUTER:
He's kindsman me as were olive, whose cy rege;
Whose all hy -livake Burry eyese feage
For will nath though up the reyal;
Whose, I ever of as clove, one, now;
Lo spence, voist, if weret, when we prove right of thy Town.

ROMEO:
Twights three is kning them broanst look bews:
We would and ostom before intenge.

LUCIO:
O;
our,
But that some is the wills graw me my brefore his sprenger.

POLY:
Yet my nour sall atonder, that the blot
But.

CORIOLANUS:
And the his spatis'd that him; I case were too kenger stage was babst.

KING HENRY VI:
O, swiff-wilty may, Gold us bid the hast her spersend,
That your our slezen thry towels too sling
sir, I knom her all sweetman, but therefore; my not thou
the ploorine son!
Theer ten, Isay decled me welp agets
What mover crive in all thau the soful.
You a wrant, but that you, here the had seingingfanctions.

BENBY:


## Bilan des quatre TP

- Les embeddings representent les tokens.
- L'attention fait circuler le contexte entre les positions.
- Le MLP transforme chaque token et mobilise des associations apprises.
- La loss (entropie croisee sur le prochain token) agit sur l'ensemble du modele
  via la retropropagation.